In [1]:
import pandas as pd
import torch
from utils import loadVocab, START_TAG, END_TAG, UNK_TAG,vocab_path

In [2]:
#Import module from CDCR folder
import sys
sys.path.append(r'C:\Users\sago\Documents\GitHub\CDCR_lexical_diversity\CDCR\data')

import dataset

In [3]:
class Config():
    def __init__(self, data_folder,mention_type,use_gold_mentions=True):
        self.data_folder = data_folder
        self.use_gold_mentions= use_gold_mentions
        self.mention_type = mention_type

In [4]:
ecbplus_config = Config(data_folder="../../CDCR/data/ECBplus-prep/output_data/",mention_type="entities_mentions",use_gold_mentions=True)

In [5]:
ecbplus = dataset.CrossEncoderDatasetFull(config=ecbplus_config)

In [6]:
ecbplus.instances[0]

"A friend of the actress told People she went to Promises [START] on Tuesday [END] and that her friends and family supported her decision . [SEP] Tara Reid has entered a rehab program , the American Pie star 's rep confirmed to People and Us Weekly [START] on Friday [END] . [SEP]"

In [7]:
ecbplus.labels[0]

tensor(1.)

In [8]:
#Modification from the original load_text_con method
def load_text_con(cdcr_dataset,max_rows):
    #Load CDCR data
    

    
    inputs = [[START_TAG] + x.strip().split(" ") + [END_TAG] for x in cdcr_dataset.instances[0:max_rows]] 
    # I could add this to the standard processing. There is unnecessary overhead
    

    # label_with_pos = [[START_TAG] + x + [SEP_TAG] + y + [END_TAG] for x, y in zip(label_sen, label_pos)]
    labels = cdcr_dataset.labels[0:max_rows]

    return inputs, labels

In [9]:
inputs, labels = load_text_con(ecbplus,1000)

In [11]:
max([len(x) for x in inputs])

99

In [18]:
#Temp vocab based on MSCOCO. I should use a pre-trained subtokenizer
#loadVocab(vocab_fpath="mscoco.vocab")

In [12]:
def load_data_tensor_con(cdcr_dataset,max_rows, vocabPath, maxlen=200):
    token2idx, _ = loadVocab(vocabPath)

    tokenized_inputs = []
    inputs, labels = load_text_con(cdcr_dataset,max_rows)
    padded_length = maxlen
    for x in inputs:
        x = [token2idx.get(t, token2idx[UNK_TAG]) for t in x] #Fills with Unknown tag value if token not found 
        # Definitely need to improve the tokenization. Gotta be careful with how to add these values to the vocabulary
        x = x + [0] * (padded_length - len(x)) 

        tokenized_inputs.append(x)
    return torch.tensor(tokenized_inputs), labels

In [13]:
tokenized_inputs, final_labels = load_data_tensor_con(cdcr_dataset=ecbplus,max_rows=1000,vocabPath=vocab_path)

In [19]:
dictionar = loadVocab(vocab_path)

In [17]:
tokenized_inputs[0][0]

tensor(2)

In [25]:
#Remember that choosing max_rows like these will result in heavily imbalanced labels given that the pairs
# Are built with a combinatorial of all clusters in a topic

In [24]:
final_labels.mean()

tensor(0.0180)

In [27]:

torch.save(tokenized_inputs, 'ecbplus_tokenized_inputs.pt')
torch.save(final_labels, 'ecbplus_labels.pt')